In [ ]:
import torch
import transformer_lens
from sae_lens import SAE

In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
sae = SAE.from_pretrained(
    release="gpt2-small-res-jb",  # Имя датасета SAE
    sae_id="blocks.6.hook_resid_pre",  # Какой слой
    device="cuda"
)

In [165]:
from transformer_lens import HookedTransformer

model = HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)


C:\Users\Danii\AppData\Local\Temp\ipykernel_18072\2643452323.py:3: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("gpt2-small",  device=device, use_cache=True)


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loaded pretrained model gpt2-small into HookedTransformer


In [166]:
import json

with open('deepseek.json') as f:
    data = json.load(f)
print(len(data))

997


In [167]:

idx = [887]
steering_vector = sae.W_dec[idx]
alpha = 0.05

In [168]:
import torch.nn.functional as F
slerp_k = []
def slerp(x, y, t, eps=1e-7):
    x = x[0, -1, :]
    y = y[0, -1, :]

    x_norm = x.norm()
    y_norm = y.norm()

    x_unit = x / x_norm.clamp_min(eps)
    y_unit = y / y_norm.clamp_min(eps)

    cos_theta = torch.dot(
        x_unit,
        y_unit,
    ).clamp(-1.0, 1.0)

    theta = torch.acos(cos_theta)
    sin_theta = torch.sin(theta)

    if sin_theta.abs() < eps:
        return x

    k_x = (
        torch.sin((1.0 - t) * theta)
        / sin_theta
    )
    k_y = (
        torch.sin(t * theta)
        / sin_theta
    )

    slerp_k.append(k_x.detach().cpu().item())

    direction = k_x * x_unit + k_y * y_unit

    # Возвращаем норму исходного hidden state
    return direction * x_norm

In [169]:
model

HookedTransformer(
  (embed): Embed()
  (hook_embed): HookPoint(name='hook_embed')
  (pos_embed): PosEmbed()
  (hook_pos_embed): HookPoint(name='hook_pos_embed')
  (blocks): TypedModuleList(
    (0): TransformerBlock(
      (ln1): LayerNormPre(
        (hook_scale): HookPoint(name='blocks.0.ln1.hook_scale')
        (hook_normalized): HookPoint(name='blocks.0.ln1.hook_normalized')
      )
      (ln2): LayerNormPre(
        (hook_scale): HookPoint(name='blocks.0.ln2.hook_scale')
        (hook_normalized): HookPoint(name='blocks.0.ln2.hook_normalized')
      )
      (attn): Attention(
        (hook_k): HookPoint(name='blocks.0.attn.hook_k')
        (hook_q): HookPoint(name='blocks.0.attn.hook_q')
        (hook_v): HookPoint(name='blocks.0.attn.hook_v')
        (hook_z): HookPoint(name='blocks.0.attn.hook_z')
        (hook_attn_scores): HookPoint(name='blocks.0.attn.hook_attn_scores')
        (hook_pattern): HookPoint(name='blocks.0.attn.hook_pattern')
        (hook_result): HookPoint(name

In [170]:
activates = []
post_activates = []
slerp_k = []
# def hook_steering_slerp(tensor, hook):
#     activates.append(tensor.detach().cpu().clone())
#     new_last_state = slerp(tensor, steering_vector.unsqueeze(0), t = 0.05)
#     result = tensor.clone()
#     result[:, -1, :] = new_last_state.unsqueeze(0)
#     post_activates.append(result.detach().cpu().clone())
#     return result

def hook_my(tensor, hook):
    print(tensor.shape)
    activates.append(tensor[:,-1,:])
    return tensor

In [171]:
texts= ['A portrait hangs on the wall.',
       'On the wall hangs a picture of the battle.',
       'On the wall hangs a poster for the movie "Star Wars".',
       'There is a picture of a cat hanging on the wall.'
       'A photo of a dog hangs on the wall.',
       'There is nothing hanging on the wall.',
       'A map of the district hangs on the wall.',
       'On the wall hangs a plan of a tank breakthrough',
       ]

In [ ]:
model.add_hook(
name="blocks.6.hook_resid_pre",
hook=hook_my,
dir="fwd"
)
logit = model(texts)



In [159]:
len(activates)

1

In [160]:
activates[0].shape

torch.Size([7, 768])

In [161]:
cos_matrix = [[] for _ in range(7)]

for x, row_x in enumerate(activates[0]):
    for y, row_y in enumerate(activates[0]):
        cos_matrix[x][y].append(torch.dot(row_x, row_y))

IndexError: list index out of range